# 02 — Model Training (NVIDIA B200)

Step B of the work plan. The notebook trains, tunes, calibrates and evaluates the fraud
model, then exports CPU-loadable artifacts. All logic lives in `src/` (`src/ml/train.py`,
`preprocess.py`, `evaluate.py`, `explain.py`); this notebook only orchestrates and plots.

**How to run on the B200 server:**

1. Upload `b200_bundle.zip` and this notebook to the same folder.
2. Edit nothing except the configuration cell, then *Run All*.
3. Download `b200_outputs_<timestamp>.zip`, written next to the notebook at the end.

Every finished model is checkpointed to `outputs/checkpoints/`. If the kernel dies, *Run All*
again: with `RESUME = True`, finished models and Optuna trials are reloaded, not retrained.

**Data rules:**

- Models are selected on `real_validation.csv`; synthetic rows are only ever used for training.
- `real_test.csv` is opened once, in section 8, and only when `FAST_MODE = False`.
- In FAST_MODE, section 8 runs on real validation as a stand-in, so the CPU smoke test never
  touches the test set.

In [ ]:
# ============================ CONFIGURATION (the only cell to edit) ============================
FAST_MODE = False        # True: 20k-row sample, few trials/epochs, CPU OK (laptop smoke test)
RUN_SECTIONS = {
    "baselines": True, "data_mix": True, "zoo": True, "tuning": True, "final_model": True,
    "final_evaluation": True, "explain": True, "save": True,
}
MODELS = ["rf", "xgboost", "lightgbm", "catboost", "mlp"]   # model zoo (LogReg is a baseline)
N_TRIALS = 80            # Optuna trials per finalist (FAST_MODE uses 3)
TUNE_TOP = 2             # tune the best N families from the zoo
SEEDS = [42, 43, 44]     # seed ensemble for the final model and stability checks
RESUME = True            # reuse checkpoints / Optuna trials from an earlier (interrupted) run
INSTALL_REQUIREMENTS = True   # pip install requirements-train.txt (never touches torch/CUDA)

In [ ]:
# --- Project root: unpack b200_bundle.zip if it sits next to this notebook -------------------
import os, sys, zipfile, subprocess, json, time, shutil
from pathlib import Path

NB_DIR = Path.cwd()
bundle = NB_DIR / "b200_bundle.zip"
if bundle.exists():
    target = NB_DIR / "b200_bundle"
    stamp = target / ".bundle_mtime"
    if not stamp.exists() or stamp.read_text() != str(bundle.stat().st_mtime):
        if target.exists():
            shutil.rmtree(target)
        zipfile.ZipFile(bundle).extractall(target)
        stamp.write_text(str(bundle.stat().st_mtime))
        print("unpacked", bundle.name)

def find_project_root() -> Path:
    for c in [NB_DIR / "b200_bundle", NB_DIR, *NB_DIR.parents]:
        if (c / "config" / "settings.yaml").exists() and (c / "src").is_dir():
            return c.resolve()
    raise FileNotFoundError("Project not found: put b200_bundle.zip next to this notebook "
                            "or open the notebook inside the repository.")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
OUT = NB_DIR / "outputs"
OUT.mkdir(exist_ok=True)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("OUTPUTS      =", OUT)
if (PROJECT_ROOT / "MANIFEST.json").exists():
    man = json.loads((PROJECT_ROOT / "MANIFEST.json").read_text())
    print("bundle commit", man["git_commit"][:10], "created", man["created_utc"])

In [ ]:
# --- Install pinned training libraries without touching the preinstalled torch/CUDA stack ----
req = PROJECT_ROOT / "requirements-train.txt"
if INSTALL_REQUIREMENTS and req.exists():
    frozen = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
    keep = [l for l in frozen.splitlines()
            if l.lower().split("==")[0].split(" @ ")[0].startswith(("torch", "nvidia-", "triton", "numpy", "cuda-"))
            and "==" in l]
    constraints = OUT / "constraints-preinstalled.txt"
    constraints.write_text("\n".join(keep) + "\n")
    print("pinning preinstalled packages:", ", ".join(k.split("==")[0] for k in keep) or "(none)")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req), "-c", str(constraints)],
                       capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-3000:])
    if r.returncode != 0:
        raise RuntimeError("pip install failed (see above). If a pinned version is unavailable for "
                           "this Python, tell Claude the error; do not upgrade torch.")
    import importlib; importlib.invalidate_caches()

In [ ]:
# --- Environment report (fails early if the GPU cannot be used) -----------------------------
from src.ml.train import environment_report, check_blackwell, detect_device
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not available")
ENV = environment_report()
for k, v in ENV.items():
    print(f"{k:>20}: {v}")
check_blackwell(ENV, require_gpu=not FAST_MODE)
DEVICE = detect_device()
print("DEVICE =", DEVICE, "| FAST_MODE =", FAST_MODE)

In [ ]:
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
from src.ml.train import TrainingRun
from src.ml import eda   # shared chart style and palette

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
LOG = OUT / "run_log.txt"
def log(msg):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(LOG, "a") as fh:
        fh.write(line + "\n")

run = TrainingRun(OUT, fast=FAST_MODE, device=DEVICE, seeds=SEEDS,
                  n_trials=3 if FAST_MODE else N_TRIALS, tune_top=TUNE_TOP, resume=RESUME, log=log)
FIG = OUT / "figures"
log(f"data: augmented train {len(run.aug):,} rows, real validation {len(run.val):,} rows "
    f"({run.y_val.sum()} frauds); max_rows={run.max_rows}")

## 1. Baselines: all-legit, rules-only (red flags), class-weighted logistic regression

In [ ]:
if RUN_SECTIONS["baselines"]:
    display(run.baselines().round(4))

## 2. Data-mix experiment (the core research result)

The same XGBoost model is trained on real data only, on real data plus 10–100% of the
synthetic rows, on real data plus all synthetic rows at sample weights 0.1, 0.3 and 1.0, and
on synthetic data only. Each mix is scored by real-validation PR-AUC (mean of 2 seeds). The
best mix is used from here on.

In [ ]:
if RUN_SECTIONS["data_mix"]:
    mix = run.data_mix("xgboost")
    display(mix.round(4))
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    m1 = mix[(mix.synthetic_weight == 1.0) & mix.include_real].sort_values("synthetic_rows")
    ax.errorbar(m1.synthetic_rows, m1.pr_auc_mean, yerr=m1.pr_auc_sd, fmt="o-", lw=2, ms=6, capsize=0,
                color=eda.ORIGIN_COLORS["real_train"], label="real + synthetic (weight 1.0)")
    mw = mix[(mix.synthetic_weight != 1.0) & mix.include_real]
    ax.errorbar(mw.synthetic_rows, mw.pr_auc_mean, yerr=mw.pr_auc_sd, fmt="s", ms=7, capsize=0,
                color=eda.ORIGIN_COLORS["synthetic"], label="real + all synthetic, down-weighted")
    for _, r in mw.iterrows():
        ax.annotate(f"w={r.synthetic_weight}", (r.synthetic_rows, r.pr_auc_mean), xytext=(6, -3),
                    textcoords="offset points", fontsize=8, color=eda.INK_2)
    so = mix[~mix.include_real]
    ax.errorbar(so.synthetic_rows, so.pr_auc_mean, yerr=so.pr_auc_sd, fmt="D", ms=6, capsize=0,
                color=eda.ORIGIN_COLORS["real_validation"], label="synthetic only")
    ax.axhline(run.y_val.mean(), color=eda.MUTED, ls="--", lw=1)
    ax.set_xscale("symlog", linthresh=10_000); ax.set_xlabel("Synthetic rows in training (symlog)")
    ax.set_ylabel("Real-validation PR-AUC"); ax.legend(frameon=False, fontsize=8)
    ax.set_title("Does synthetic data help? (XGBoost; dashed = random scorer)", loc="left", fontsize=10)
    ax.grid(color=eda.GRID, lw=0.6); eda.style_axes(ax)
    eda.save(fig, FIG / "data_mix.png"); display(Image(FIG / "data_mix.png"))

## 3. Model zoo (class-weighted; GPU for XGBoost, CatBoost and the MLP)

In [ ]:
if RUN_SECTIONS["zoo"]:
    zoo = run.zoo(MODELS)
    display(zoo.round(4))
    t = zoo[~zoo.model.str.startswith("mix_")]
    fig, ax = plt.subplots(figsize=(7.5, 0.42 * len(t) + 1.2))
    eda.plot_dots(ax, t.model.tolist(), {"PR-AUC (real validation)": (t.pr_auc, t.pr_lo, t.pr_hi)},
                  "PR-AUC with 95% bootstrap CI", ref=run.y_val.mean())
    ax.set_title("Model comparison on real validation (dashed = random scorer)", loc="left", fontsize=10)
    eda.save(fig, FIG / "model_comparison.png"); display(Image(FIG / "model_comparison.png"))

## 4. Optuna tuning of the best families

TPE sampling maximises real-validation PR-AUC. To avoid picking a lucky trial with only 139
validation frauds, the top 5 trials are each refitted with 3 seeds, and the configuration with
the best **mean** score is kept. Trials are stored in `outputs/optuna.db`, so an interrupted
run resumes where it stopped.

In [ ]:
if RUN_SECTIONS["tuning"]:
    display(run.tune_best().round(4))

## 5. Final model: seed ensemble with cross-fitted calibration and cost-based thresholds

In [ ]:
if RUN_SECTIONS["final_model"]:
    final = run.finalize()
    cal = final["calibration"]
    print(f"calibration: {cal['method']} | Brier uncalibrated {cal['uncalibrated_brier']:.4f}, "
          f"Platt {cal['sigmoid']['brier']:.4f}, isotonic {cal['isotonic']['brier']:.4f} (out-of-fold)")
    display(pd.DataFrame(final["thresholds"]).T.round(4))
    print("risk bands:", final["risk_bands"])

    from src.ml.evaluate import reliability_table
    rel = reliability_table(run.y_val, final["oof_cal"])
    fig, ax = plt.subplots(figsize=(5, 4.5))
    lim = max(rel.mean_pred.max(), rel.observed.max()) * 1.1
    ax.plot([0, lim], [0, lim], color=eda.MUTED, ls="--", lw=1)
    ax.errorbar(rel.mean_pred, rel.observed, yerr=np.clip([rel.observed - rel.lo, rel.hi - rel.observed], 0, None),
                fmt="o-", color=eda.ORIGIN_COLORS["real_train"], lw=2, ms=5, capsize=0)
    ax.set_xlabel("Mean predicted fraud probability"); ax.set_ylabel("Observed fraud rate (95% CI)")
    ax.set_title(f"Reliability on real validation (out-of-fold, {cal['method']})", loc="left", fontsize=10)
    ax.grid(color=eda.GRID, lw=0.6); eda.style_axes(ax)
    eda.save(fig, FIG / "reliability.png"); display(Image(FIG / "reliability.png"))

## 6. Decision policy

| Condition | Recommendation |
|---|---|
| score ≥ `high` band (10:1 cost threshold) | **FLAG_FOR_INVESTIGATION** |
| blocking validation failure (e.g. missing age) | **REQUEST_MORE_INFO** |
| otherwise | **APPROVE**, always subject to adjuster review |

The 5:1 and 20:1 thresholds and the top-10% capacity threshold are reported alongside, in
`thresholds.csv` and `model_card.json`.

## 7. Saving checkpoints before the test set is opened

In [ ]:
if RUN_SECTIONS["save"] and RUN_SECTIONS["final_model"]:
    run.save_artifacts()     # saved now so a crash in later sections never loses the model

## 8. Final held-out evaluation (real test, opened once; validation stand-in in FAST_MODE)

In [ ]:
if RUN_SECTIONS["final_evaluation"]:
    SPLIT = "real_validation" if FAST_MODE else "real_test"
    ev = run.evaluate(SPLIT)
    log(f"evaluated on {SPLIT}: n={ev['n']}, frauds={ev['fraud']}")
    display(ev["metrics"].round(4))
    display(ev["comparison"].round(4))
    s = ev["success"]
    print(f"Success criterion (precision and recall both meaningfully above naive): {'MET' if s['met'] else 'NOT MET'}\n"
          f"  precision lower CI {s['precision_lo']:.3f} vs random-flagging precision {s['random_precision']:.3f}\n"
          f"  recall lower CI {s['recall_lo']:.3f} vs random recall at the same flag rate {s['random_recall_at_same_flag_rate']:.3f}")
    display(ev["lift"].round(3))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    lt = ev["lift"]
    axes[0].bar(lt.decile, lt.lift, color=eda.ORIGIN_COLORS["real_train"], width=0.7)
    axes[0].axhline(1, color=eda.MUTED, ls="--", lw=1)
    axes[0].set_xlabel("Score decile (1 = highest)"); axes[0].set_ylabel("Lift over base rate")
    axes[0].set_title(f"Lift by decile ({SPLIT})", loc="left", fontsize=10); eda.style_axes(axes[0])
    axes[1].plot(np.r_[0, lt.decile / 10], np.r_[0, lt.cumulative_recall], "o-", lw=2,
                 color=eda.ORIGIN_COLORS["real_train"], label="model")
    axes[1].plot([0, 1], [0, 1], color=eda.MUTED, ls="--", lw=1, label="random")
    axes[1].set_xlabel("Share of claims investigated"); axes[1].set_ylabel("Share of frauds caught")
    axes[1].legend(frameon=False, fontsize=8); axes[1].set_title("Cumulative gains", loc="left", fontsize=10)
    axes[1].grid(color=eda.GRID, lw=0.6); eda.style_axes(axes[1])
    eda.save(fig, FIG / "lift.png"); display(Image(FIG / "lift.png"))

## 9. Explainability (SHAP), error analysis and fairness

In [ ]:
if RUN_SECTIONS["explain"] and RUN_SECTIONS["final_evaluation"]:
    sh = run.explain()
    imp = sh["importance"].head(15)
    fig, ax = plt.subplots(figsize=(7, 5))
    eda.plot_dots(ax, imp.index.tolist(), {"mean |SHAP|": (imp.values, None, None)}, "Mean |SHAP| (log-odds)")
    ax.set_title("Global feature importance (SHAP, uncalibrated base model)", loc="left", fontsize=10)
    eda.save(fig, FIG / "shap_global.png"); display(Image(FIG / "shap_global.png"))

    from src.ml.explain import describe_feature
    names = sh["names"]; X = np.asarray(sh["X"], dtype=float) if run.final["family"] != "catboost" else None
    top_feats = [n for n in names if describe_feature(n)[0].split(" = ")[0] in imp.index[:4].tolist()][:4]
    if X is not None and top_feats:
        fig, axes = plt.subplots(1, len(top_feats), figsize=(4 * len(top_feats), 3.4), squeeze=False)
        for ax, fname in zip(axes[0], top_feats):
            j = names.index(fname)
            ax.scatter(X[:, j] + np.random.default_rng(0).normal(0, 0.03, len(X)), sh["values"][:, j], s=6,
                       alpha=0.4, color=eda.ORIGIN_COLORS["real_train"])
            ax.axhline(0, color=eda.MUTED, lw=1); ax.set_xlabel(fname); ax.set_ylabel("SHAP (log-odds)")
            eda.style_axes(ax)
        fig.suptitle("SHAP dependence for the top features", x=0.01, ha="left", fontsize=10)
        fig.tight_layout(); eda.save(fig, FIG / "shap_dependence.png"); display(Image(FIG / "shap_dependence.png"))
    display(sh["local"])

In [ ]:
if RUN_SECTIONS["final_evaluation"]:
    err = run.error_analysis()
    print("Segments where frauds are missed most (n >= 50, recall lowest):")
    display(err[(err.n >= 50) & (err.fraud > 0)].sort_values("recall").head(8).round(3))
    print("Segments most over-flagged (highest false-positive rate, n >= 50):")
    display(err[err.n >= 50].sort_values("fpr", ascending=False).head(8).round(3))
    fair = run.fairness()
    display(fair.round(3))
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
    for ax, metric in zip(axes, ["recall", "fpr", "flag_rate"]):
        t = fair[fair.n >= 30]
        eda.plot_dots(ax, (t.attribute + ": " + t.group.astype(str) + "  (n=" + t.n.astype(str) + ")").tolist(),
                      {metric: (t[metric], None, None)}, metric)
        ax.set_title(f"{metric} by group", loc="left", fontsize=10)
    fig.tight_layout(); eda.save(fig, FIG / "fairness.png"); display(Image(FIG / "fairness.png"))

## 10. Save artifacts and package the outputs for download

In [ ]:
if RUN_SECTIONS["save"] and RUN_SECTIONS["final_model"]:
    run.save_artifacts()     # re-save with the evaluation, SHAP explainer and final model card
    print(json.dumps({k: run.card[k] for k in ("model_version", "family", "risk_bands", "fast_mode")}, indent=2))
    # Verify the artifacts load and score on CPU, exactly as on the laptop
    from src.ml.predict import load_artifacts, score_claims
    load_artifacts.cache_clear()
    art = load_artifacts(str(OUT / "models"))
    print(score_claims(run.val.head(3), art))
zip_path = run.zip_outputs()
print("DOWNLOAD THIS FILE:", zip_path)